# Looking at what is in the image

The previous page established that the file reads, what type the numbers are, and how
big it is.  Three questions remain before the scene can be used.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/readcheck.py', 'lib/quality.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'readcheck', 'quality'):
    sys.modules.pop(m, None)
import rasterio
from quality import *
from readcheck import check_bands, check_order

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
PATH = get('dataset/intake/input.tif', 'input.tif')
CLEAR = get('dataset/condition/clear.tif', 'clear.tif')
CLOUDY = get('dataset/condition/cloudy.tif', 'cloudy.tif')

## Q1.  What does the distribution look like?

Percentiles first, then the shape behind them.

In [ ]:
distribution(PATH)

In [ ]:
show_distribution(PATH)

Half the pixels sit below 321 and 98 % below 738, but the range runs to 4095.  The
distribution is not centred — it is a narrow peak with a long thin tail.

That is the normal shape for a satellite scene, and it is why a display range has to be
chosen.  Stretching to `min - max` would spend the whole screen on a tail that holds
2 % of the pixels.

## Q2.  How are the bands arranged?

Ask the file first.

In [ ]:
with rasterio.open(PATH) as d:
    print('bands      ', d.count, d.indexes)
    print('names      ', d.descriptions)
    print('colorinterp', tuple(c.name for c in d.colorinterp))

No names, and `colorinterp` says `gray` followed by `undefined` three times.  **The
file does not know which band is which.**  That has to come from the product spec — or
from looking.

In [ ]:
check_bands(PATH)

In the fourth band the river goes black and the parks go bright, which is what water
and vegetation do to near infrared.  So band 4 is NIR and bands 1 - 3 are red, green,
blue.

Getting it wrong raises no error.

In [ ]:
check_order(PATH)

## Q3.  What condition is the scene in?

Cloud and sharpness decide whether the scene is worth processing at all.  Neither needs
a model to spot — two scenes from the same sensor, side by side.

In [ ]:
condition([CLEAR, CLOUDY], ['clear', 'cloudy'])

In [ ]:
show_condition([CLEAR, CLOUDY], ['clear', 'cloudy'])

The cloudy scene is brighter everywhere (mean 823 against 305), a third of it is above
900, and its sharpness is **five times lower** — 0.0137 against 0.0691.

Sharpness here is the mean gradient divided by the mean level, so brightness cancels
out and a dark scene can be compared with a bright one.  Cloud is smooth, so it drives
that number down.  These are indicators, not a cloud mask, but they separate a usable
scene from a covered one without anyone having to look.